In [1]:
!pip install lightly #good library for contrastive learning based on google search

In [2]:
%%writefile cub2011.py
import os
import pandas as pd
from torchvision.datasets.folder import default_loader
from torchvision.datasets.utils import download_url
from torch.utils.data import Dataset


class Cub2011(Dataset):
    base_folder = 'CUB_200_2011/images'
    url = 'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz'
    filename = 'CUB_200_2011.tgz'
    tgz_md5 = '97eceeb196236b17998738112f37df78'

    def __init__(self, root, train=True, transform=None, loader=default_loader, download=True):
        self.root = os.path.expanduser(root)
        self.transform = transform
        self.loader = default_loader
        self.train = train

        if download:
            self._download()

        if not self._check_integrity():
            raise RuntimeError('Dataset not found or corrupted.' +
                               ' You can use download=True to download it')

    def _load_metadata(self):
        images = pd.read_csv(os.path.join(self.root, 'CUB_200_2011', 'images.txt'), sep=' ',
                             names=['img_id', 'filepath'])
        image_class_labels = pd.read_csv(os.path.join(self.root, 'CUB_200_2011', 'image_class_labels.txt'),
                                         sep=' ', names=['img_id', 'target'])
        train_test_split = pd.read_csv(os.path.join(self.root, 'CUB_200_2011', 'train_test_split.txt'),
                                       sep=' ', names=['img_id', 'is_training_img'])

        data = images.merge(image_class_labels, on='img_id')
        self.data = data.merge(train_test_split, on='img_id')

        if self.train:
            self.data = self.data[self.data.is_training_img == 1]
        else:
            self.data = self.data[self.data.is_training_img == 0]

    def _check_integrity(self):
        try:
            self._load_metadata()
        except Exception:
            return False

        for index, row in self.data.iterrows():
            filepath = os.path.join(self.root, self.base_folder, row.filepath)
            if not os.path.isfile(filepath):
                print(filepath)
                return False
        return True

    def _download(self):
        import tarfile

        if self._check_integrity():
            print('Files already downloaded and verified')
            return

        download_url(self.url, self.root, self.filename, self.tgz_md5)

        with tarfile.open(os.path.join(self.root, self.filename), "r:gz") as tar:
            tar.extractall(path=self.root)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        path = os.path.join(self.root, self.base_folder, sample.filepath)
        target = sample.target - 1  # Targets start at 1 by default, so shift to 0
        img = self.loader(path)

        if self.transform is not None:
            img = self.transform(img)

        return img, target

Overwriting cub2011.py


In [3]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from lightly.models.modules import heads
from lightly.loss import NTXentLoss
from lightly.transforms import SimCLRTransform
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from cub2011 import Cub2011
import torchvision.models as models
import random
from tqdm import tqdm
import sklearn.preprocessing
import logging
import sklearn.cluster
import sklearn.metrics.cluster

# Transform pipeline
Apply random crop, horizontal flip, color jitter, grayscale. Blur, then return as tensor.

In [4]:
simclr_transform = T.Compose([
    T.RandomResizedCrop(32),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Dataset
Create two views (xi, xj) of the given image from the dataset.

In [5]:
class SimCLRDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.dataset = base_dataset
        self.transform = transform

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        xi = self.transform(image)
        xj = self.transform(image)
        return xi, xj

    def __len__(self):
        return len(self.dataset)

# SimCLR Model
- Encoder: ResNet18, ResNet50, ResNet101
- Projection Head: maps features from encoder to 128-Dimensional Contrastive Space
- Output: contrastive loss $z$

In [6]:
class SimCLRModel(nn.Module):
    def __init__(self, model_type, projection_dim=128):
        super().__init__()
        base_model = model_type(weights=None)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential(
            nn.Linear(num_ftrs, 2048),
            nn.ReLU(),
            nn.Linear(2048, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

# Loss Function
NT-Xent

In [7]:
def nt_xent_loss(z_i, z_j, temperature=0.05):
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T)
    N = z_i.shape[0]

    mask = (~torch.eye(2*N, dtype=bool)).to(z.device)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask

    positive_sim = torch.exp(F.cosine_similarity(z_i, z_j) / temperature)
    positives = torch.cat([positive_sim, positive_sim], dim=0)

    denominator = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denominator)
    return loss.mean()

ProxyNCA

In [7]:
def binarize_and_smooth_labels(T, nb_classes, smoothing_const=0.1):
    T = T.cpu().numpy()
    T = sklearn.preprocessing.label_binarize(T, classes=range(nb_classes))
    T = T * (1 - smoothing_const)
    T[T == 0] = smoothing_const / (nb_classes - 1)
    return torch.FloatTensor(T).to(device)


class ProxyNCA(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, smoothing_const=0.1,
                 scaling_x=3.0, scaling_p=3.0):
        super().__init__()
        self.proxies = nn.Parameter(torch.randn(num_classes, embedding_dim) / 8)
        self.smoothing_const = smoothing_const
        self.scaling_x = scaling_x
        self.scaling_p = scaling_p
        self.num_classes = num_classes

    def forward(self, z, labels):
        P = F.normalize(self.proxies, p=2, dim=-1) * self.scaling_p
        Z = F.normalize(z, p=2, dim=-1) * self.scaling_x

        D = torch.cdist(Z, P) ** 2

        T = binarize_and_smooth_labels(labels, self.num_classes, self.smoothing_const)
        loss = torch.sum(-T * F.log_softmax(-D, dim=-1), dim=-1)
        return loss.mean()

# Training Loop

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = Cub2011(root='./cub2011', train=True, download=True)
contrastive_dataset = SimCLRDataset(train_dataset, simclr_transform)
train_loader = DataLoader(contrastive_dataset, batch_size=1024, shuffle=True, num_workers=2)
num_epochs = 200

for model_type in [(models.resnet18, "resnet18"), (models.resnet50, "resnet50"), (models.resnet101, "resnet101")]:
#for model_type in [(models.resnet50, "resnet50")]:
  model = SimCLRModel(model_type[0]).to(device)

  #load model from checkpoint
  #model.load_state_dict(torch.load("/content/test_model_resnet18.pth", map_location=device))

  optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)

  for epoch in range(num_epochs):
      model.train()
      total_loss = 0
      for x_i, x_j in tqdm(train_loader):
          x_i, x_j = x_i.to(device), x_j.to(device)
          z_i = model(x_i)
          z_j = model(x_j)

          loss = nt_xent_loss(z_i, z_j)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()
          total_loss += loss.item()

      print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

  #torch.save(model.state_dict(), "/content/drive/MyDrive/Colab/COMP597FinalProject/test_model_" + model_type[1] + "_200ep.pth")
  torch.save(model.state_dict(), "C:\\Users\\Haytham\\Desktop\\comp597final\\COMP597_FINALHZ_MODEL_" + model_type[1] + "_200ep.pth")

100%|██████████| 1.15G/1.15G [00:35<00:00, 32.0MB/s]


KeyboardInterrupt: 

In [ ]:
#training loop for proxynca

#transform images for proxy
proxy_transform = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

full_train_dataset = Cub2011(root='./cub2011', train=True, download=True, transform=proxy_transform)
# Only keep samples with target in 0–99 (first 100 classes)
train_indices = [i for i, (_, target) in enumerate(full_train_dataset) if 0 <= target < 100]
from torch.utils.data import Subset
proxy_train_dataset = Subset(full_train_dataset, train_indices)
proxy_train_loader = DataLoader(proxy_train_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

#just use resnet50 for now
for model_type in [(models.resnet50, "resnet50")]:
    base = model_type[0](weights=models.ResNet50_Weights.DEFAULT)
    embedding_dim = 64
    base.fc = nn.Linear(base.fc.in_features, embedding_dim)
    encoder = base.to(device)

    proxy_loss_fn = ProxyNCA(
        num_classes=100,  #100 classes for training split
        embedding_dim=embedding_dim,
    ).to(device)

    optimizer = torch.optim.Adam([
    {'params': encoder.parameters(), 'lr': 1e-4},
    {'params': proxy_loss_fn.parameters(), 'lr': 1.0}
], weight_decay=1e-4)

    for epoch in range(50):
        encoder.train()
        proxy_loss_fn.train()
        total_loss = 0
        for images, labels in tqdm(proxy_train_loader):
            images, labels = images.to(device), labels.to(device)
            z = encoder(images)

            loss = proxy_loss_fn(z, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch+1} | Loss: {total_loss / len(proxy_train_loader):.4f}")

    torch.save({'encoder': encoder.state_dict(),'proxies': proxy_loss_fn.state_dict()}, "C:\\Users\\Haytham\\Desktop\\comp597final\\proxynca_model_" + model_type[1] + ".pth")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 160MB/s]
100%|██████████| 23/23 [00:29<00:00,  1.28s/it]


epoch 1 | Loss: 4.4754


100%|██████████| 23/23 [00:28<00:00,  1.25s/it]


epoch 2 | Loss: 2.2039


...

epoch 48 | Loss: 1.1369


100%|██████████| 23/23 [00:33<00:00,  1.46s/it]


epoch 49 | Loss: 1.1403


100%|██████████| 23/23 [00:32<00:00,  1.42s/it]

epoch 50 | Loss: 1.1367


In [8]:
import os
save_path = os.path.join(os.getcwd(), 'proxynca_model_resnet50.pth')
torch.save({
    'encoder': encoder.state_dict(),
    'proxies': proxy_loss_fn.state_dict()
}, save_path)
print(f"Model saved to: {save_path}")

from google.colab import files
files.download(save_path)

NameError: name 'encoder' is not defined

In [16]:
import shutil
shutil.copy('/content/proxynca_model_resnet50.pth', '/content/drive/MyDrive/proxynca_model_resnet50.pth')
print("Model saved to Google Drive: My Drive/proxynca_model_resnet50.pth")

Model saved to Google Drive: My Drive/proxynca_model_resnet50.pth


### Evaluation - NT-XENT

In [5]:
#NT-Xent Eval | Resnet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimCLRModel(models.resnet18, projection_dim=128).to(device)
model.load_state_dict(torch.load("/content/test_model_resnet50.pth", map_location=device))

#similar transformation t o training transformation except without the random variations
evaluation_transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])

#load dataset
test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform = evaluation_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
print(model)
model.eval()

NameError: name 'SimCLRModel' is not defined

In [ ]:
#function to extract the image embeddings of entire test dataset, creating the evaluation vector space
def extract_embeddings_labels(model, loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    embeddings, labels = [], []
    #taken from g4g
    with torch.no_grad():
      for x, y in loader:
            x = x.to(device)
            h = model.encoder(x)
            embeddings.append(h.cpu())
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

In [ ]:
test_embeddings, test_labels = extract_embeddings_labels(model, test_loader)
print(test_embeddings)

correct = 0
for i in range(len(test_labels)):
  #get current embedding
  current_embedding = test_embeddings[i]
  #calculate distance from current embedding to all embeddings
  #https://discuss.pytorch.org/t/k-nearest-neighbor-in-pytorch/59695
  #euclidean distance
  dist = torch.norm(test_embeddings - current_embedding, dim=1, p=None)
  #exclude current image from k_nn calculation
  dist[i] = float('inf')

  #get k nearest neighbors
  k_nearest = dist.topk(3, largest=False)

  #check if any of the nearest neighbors have the same label as the current test image
  for j in range(len(k_nearest.indices)):
    if (test_labels[k_nearest.indices[j]] == test_labels[i]):
      correct += 1

print(correct / len(test_labels))

Proxy-NCA Eval

In [9]:
evaluation_transform_NCA = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

def calc_recall_at_k(T, Y, k):
    """
    T : [nb_samples] (target labels)
    Y : [nb_samples x k] (k predicted labels/neighbours)
    """
    s = sum([1 for t, y in zip(T, Y) if t in y[:k]])
    return s / (1. * len(T))

def assign_by_euclidian_at_k(X, T, k):
    """
    X : [nb_samples x nb_features], e.g. 100 x 64 (embeddings)
    k : for each sample, assign target labels of k nearest points
    """
    distances = torch.cdist(X, X)
    # get nearest points
    indices = distances.topk(k + 1, largest=False)[1][:, 1: k + 1]
    return np.array([[T[i] for i in ii] for ii in indices])

def cluster_by_kmeans(X, nb_clusters):
    """
    xs : embeddings with shape [nb_samples, nb_features]
    nb_clusters : in this case, must be equal to number of classes
    """
    return sklearn.cluster.KMeans(nb_clusters).fit(X).labels_

def calc_normalized_mutual_information(ys, xs_clustered):
    return sklearn.metrics.cluster.normalized_mutual_info_score(xs_clustered, ys)

def predict_batchwise(model, dataloader):
    model_is_training = model.training
    model.eval()
    ds = dataloader.dataset
    A = [[] for i in range(len(ds[0]))]
    with torch.no_grad():
        for batch in dataloader:
            for i, J in enumerate(batch):
                if i == 0:
                    J = J.to(list(model.parameters())[0].device)
                    J = model(J).cpu()
                for j in J:
                    A[i].append(j)
    model.train()
    model.train(model_is_training)
    return [torch.stack(A[i]) for i in range(len(A))]

def evaluate(model, dataloader, with_nmi = True):
    nb_classes = dataloader.dataset.nb_classes()
    X, T, *_ = predict_batchwise(model, dataloader)
    if with_nmi:
        nmi = calc_normalized_mutual_information(
            T,
            cluster_by_kmeans(
                X, nb_classes
            )
        )
        print("NMI: {:.3f}".format(nmi * 100))
    Y = assign_by_euclidian_at_k(X, T, 8)
    Y = torch.from_numpy(Y)
    recall = []
    for k in [1, 2, 4, 8]:
        r_at_k = calc_recall_at_k(T, Y, k)
        recall.append(r_at_k)
        print("R@{} : {:.3f}".format(k, 100 * r_at_k))
    if with_nmi:
        return recall, nmi
    else:
        return recall


In [10]:
import time
def get_uploaded_file_path(uploader):
    if uploader.value:
        for filename, fileinfo in uploader.value.items():
            with open(filename, 'wb') as f:
                f.write(fileinfo['content'])
            return filename
    return None

import ipywidgets as widgets
from IPython.display import display
import torch

uploader = widgets.FileUpload(accept='.pth', multiple=False)
display(uploader)


FileUpload(value={}, accept='.pth', description='Upload')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load full test dataset
full_test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform=evaluation_transform_NCA)
# Only keep samples with target in 100–199 (last 100 classes)
test_indices = [i for i, (_, target) in enumerate(full_test_dataset) if 100 <= target < 200]
from torch.utils.data import Subset
test_dataset = Subset(full_test_dataset, test_indices)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

# Initialize the base encoder model (ResNet50 with the same fc as training)
base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
embedding_dim = 64
base_model.fc = nn.Linear(base_model.fc.in_features, embedding_dim)
encoder_model = base_model.to(device)

# Initialize the ProxyNCA loss function (num_classes=100 for test split)
proxy_nca_loss_fn = ProxyNCA(num_classes=100, embedding_dim=embedding_dim).to(device)


# Wait for upload
while not uploader.value:
    time.sleep(1)
model_path = get_uploaded_file_path(uploader)
print(f"Model uploaded to: {model_path}")

# Load the saved state dictionaries for both encoder and proxies
checkpoint = torch.load(model_path, map_location=device)
encoder_model.load_state_dict(checkpoint['encoder'])
proxy_nca_loss_fn.load_state_dict(checkpoint['proxies'])

# Set models to evaluation mode
encoder_model.eval()
proxy_nca_loss_fn.eval()

with torch.no_grad():
    logging.info("**Evaluating...**")
    # Pass the encoder_model for evaluation, as it produces the embeddings
    evaluate(encoder_model, test_loader)


Files already downloaded and verified


FileUpload(value={}, accept='.pth', description='Upload')

KeyboardInterrupt: 

In [ ]:
#NCA Eval | Resnet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ProxyNCA(num_classes=100, embedding_dim=embedding_dim).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab/COMP597FinalProject/test_model_resnet50_200ep.pth", map_location=device))

#similar transformation t o training transformation except without the random variations
evaluation_transform_NCA = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
]) 

#load dataset
test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform = evaluation_transform_NCA)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
print(model)
model.eval()

In [ ]:
#function to extract the image embeddings of entire test dataset, creating the evaluation vector space
def extract_embeddings_labels(model, loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    embeddings, labels = [], []
    #taken from g4g
    with torch.no_grad():
      for x, y in loader:
            x = x.to(device)
            h = model.encoder(x)
            embeddings.append(h.cpu())
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

In [ ]:
test_embeddings, test_labels = extract_embeddings_labels(model, test_loader)
print(test_embeddings)

correct = 0
for i in range(len(test_labels)):
  #get current embedding
  current_embedding = test_embeddings[i]
  #calculate distance from current embedding to all embeddings
  #https://discuss.pytorch.org/t/k-nearest-neighbor-in-pytorch/59695
  #euclidean distance
  dist = torch.norm(test_embeddings - current_embedding, dim=1, p=None)
  #exclude current image from k_nn calculation
  dist[i] = float('inf')

  #get k nearest neighbors
  k_nearest = dist.topk(3, largest=False)

  #check if any of the nearest neighbors have the same label as the current test image
  for j in range(len(k_nearest.indices)):
    if (test_labels[k_nearest.indices[j]] == test_labels[i]):
      correct += 1

print(correct / len(test_labels))